In [29]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import norm
import scipy.stats as sts
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [30]:
data = pd.read_csv('processed_data_tennis_scaled_v2.csv')
data

,has_secondary_sport_education,has_higher_sport_education,sport_master,sport_master_kandidate,is_champion,work_experience,master_id,rang_id,docs_verified,rating,...,photos_count,is_male,has_price,years_on_profi_ru,price_in_thousands,cur_score_percents,work_experience_sq,reputation_density,reputation_volume_interaction,champion_experience_inter
0,1,0,0,0,0,4.0,https://profi.ru/profile/AbelkhairovaSV/,86.0,1,5.0,...,10,0,1,2.416667,3.2,0.504935,16.0,1.463415,8.958797,0.000000
1,1,0,0,0,0,19.0,https://profi.ru/profile/AbramovMB/,111.0,0,0.0,...,0,1,0,0.166667,NaN,0.504337,361.0,0.000000,0.000000,0.000000
2,0,1,1,0,1,8.0,https://profi.ru/profile/AbyasovaAN/,100.0,0,0.0,...,0,0,0,2.583333,NaN,0.503504,64.0,0.000000,0.000000,2.583333
3,1,0,0,0,0,2.0,https://profi.ru/profile/AfanasyevBA4/,144.0,1,0.0,...,8,1,1,1.250000,2.0,0.503303,4.0,0.000000,0.000000,0.000000
4,1,0,1,0,0,3.0,https://profi.ru/profile/AgureyevaKM2/,75.0,1,5.0,...,0,0,1,0.666667,3.0,0.505121,9.0,2.400000,8.047190,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
407,1,0,0,1,0,5.0,https://profi.ru/profile/ZhirkovaVA10/,136.0,0,0.0,...,2,0,1,0.166667,3.0,0.500943,25.0,0.000000,0.000000,0.000000
408,1,0,0,0,0,13.0,https://profi.ru/profile/ZhuchkovaVS3/,136.0,0,0.0,...,0,0,0,0.750000,NaN,0.500192,169.0,0.000000,0.000000,0.000000
409,1,0,1,0,1,9.0,https://profi.ru/profile/ZhukovMA78/,103.0,1,0.0,...,3,1,0,0.083333,NaN,0.504611,81.0,0.000000,0.000000,0.083333
410,0,1,1,0,1,4.0,https://profi.ru/profile/ZykovaDS2/,84.0,0,5.0,...,2,0,1,2.750000,5.0,0.504794,16.0,0.266667,3.465736,2.750000


In [31]:
## делаю безлайн только на данных с ценой
data_priced = data[data['has_price'] == 1].copy()

In [32]:
y = data_priced['price_in_thousands']
X = sm.add_constant(data_priced[['has_secondary_sport_education', 'has_higher_sport_education',
       'sport_master', 'sport_master_kandidate', 'is_champion',
       'work_experience', 'rang_id', 'docs_verified', 'rating',
       'reviews_count', 'is_recomended', 'photos_count', 'is_male', 'years_on_profi_ru', 'cur_score_percents', 'work_experience_sq', 'reputation_density',
       'reputation_volume_interaction', 'champion_experience_inter']])

In [33]:
# Расчет VIF для изначальных данных
vif_data = pd.DataFrame()
vif_data["Переменная"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif_data)

                       Переменная         VIF
0                           const  284.383586
1   has_secondary_sport_education    8.707307
2      has_higher_sport_education    9.082903
3                    sport_master    1.858797
4          sport_master_kandidate    1.238310
5                     is_champion    3.168067
6                 work_experience    9.997958
7                         rang_id    1.740212
8                   docs_verified    1.264301
9                          rating    3.777690
10                  reviews_count    5.448035
11                  is_recomended    1.498524
12                   photos_count    1.152380
13                        is_male    1.300049
14              years_on_profi_ru    2.843480
15             cur_score_percents    1.086843
16             work_experience_sq    9.116383
17             reputation_density    5.422149
18  reputation_volume_interaction   11.181601
19      champion_experience_inter    3.238861


### Логика такая - для `reputation_volume_interaction`  VIF слишком большой, его удалим


### Видим, что почти у всех есть какое-то образование, признаки сильно коррелируют, так что удалим  признак `has_higher_sport_education`

In [34]:
(X['has_secondary_sport_education'] + X['has_higher_sport_education']).mean()

np.float64(0.9655172413793104)

### `rang_id`, `cur_score_percents` уберем, т.к признаки не подходят по смыслу(экономическая интуиция)

In [35]:
data_priced.drop(['has_secondary_sport_education', 'reputation_volume_interaction', 'rang_id', 'cur_score_percents'], axis=1, inplace=True)

In [36]:
model = sm.OLS(y, X).fit()

beta_exp = model.params['work_experience']
beta_exp2 = model.params['work_experience_sq']

peak = -beta_exp / (2 * beta_exp2)
## центрирование данных
data_priced['exp_centered'] = data_priced['work_experience'] - peak
data_priced['exp_centered2'] = data_priced['exp_centered'] ** 2

y = data_priced['price_in_thousands']
X = sm.add_constant(data_priced[['has_higher_sport_education',
       'sport_master', 'sport_master_kandidate', 'is_champion',
       'exp_centered', 'docs_verified', 'rating',
       'reviews_count', 'is_recomended', 'photos_count', 'is_male', 'years_on_profi_ru', 'exp_centered2', 'reputation_density',
       'champion_experience_inter']])

In [37]:
# Расчет VIF для центрированных данных
vif_data = pd.DataFrame()
vif_data["Переменная"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif_data)

                    Переменная        VIF
0                        const  12.337090
1   has_higher_sport_education   1.214726
2                 sport_master   1.754891
3       sport_master_kandidate   1.201156
4                  is_champion   3.128500
5                 exp_centered   1.524714
6                docs_verified   1.211409
7                       rating   1.549468
8                reviews_count   4.507510
9                is_recomended   1.305600
10                photos_count   1.124300
11                     is_male   1.082138
12           years_on_profi_ru   2.620070
13               exp_centered2   1.331674
14          reputation_density   4.340339
15   champion_experience_inter   3.184013


###  Вывод: все VIF меньше 5, с мультиколлинеарность не сильная

### Сделаем фичу выбросов

In [38]:
# Вдохновлено семинаром
influence = model.get_influence()
inf_sum = influence.summary_frame()
student_resid = influence.resid_studentized_external

# Рассчитываем критическое значение t-распределения для 5% уровня значимости
df = len(data_priced) - 7
crit = sts.t(df=df).ppf(0.975)
print(f"Критическое значение t-распределения: {crit}")

results = pd.concat([data_priced, inf_sum], axis=1)
high_stud_res = results[abs(student_resid) > crit]
print(f"Цены для тренеров-выбросов: {high_stud_res['price_in_thousands'].values}")
high_stud_res

# Создаем дамми-переменную на выбросы
data_priced['is_outlier'] = 0
data_priced.loc[high_stud_res.index, 'is_outlier'] = 1

print(f"Количество выбросов: {data_priced['is_outlier'].sum()}")
print(f"Доля выбросов: {round(data_priced['is_outlier'].mean()*100, 2)}%")

Критическое значение t-распределения: 1.9675964973875208
Цены для тренеров-выбросов: [10.   5.5  7.   2.  10.   6.   6.   6.  10.   2.   7. ]
Количество выбросов: 11
Доля выбросов: 3.45%


In [39]:
y = data_priced['price_in_thousands']
X = sm.add_constant(data_priced[['has_higher_sport_education',
       'sport_master', 'sport_master_kandidate', 'is_champion',
       'exp_centered', 'docs_verified', 'rating',
       'reviews_count', 'is_recomended', 'photos_count', 'is_male', 'years_on_profi_ru', 'exp_centered2', 'reputation_density',
       'champion_experience_inter', 'is_outlier']])

## Уберем самые незначимые фичи по p-value, с учетом того, что после удаления фичи AIC и BIC должны вырасти

In [40]:
## запишем колонки, которые будем дропать
cols_to_drop = []

In [41]:
base_model_res = sm.OLS(y, X).fit()
worst_p = base_model_res.pvalues.drop('const').max()
worst_var = base_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии базовой модели
print(f"Текущий AIC: {base_model_res.aic:.2f} | BIC: {base_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 979.40 | BIC: 1043.41
Худшая переменная: 'rating' (p-value = 0.8852)
------------------------------


In [42]:
base_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.331
Model:                            OLS   Adj. R-squared:                  0.295
Method:                 Least Squares   F-statistic:                     9.325
Date:                Thu, 07 May 2026   Prob (F-statistic):           9.50e-19
Time:                        19:09:00   Log-Likelihood:                -472.70
No. Observations:                 319   AIC:                             979.4
Df Residuals:                     302   BIC:                             1043.
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.7839      0.215     12.934      0.000       2.360       3.207
has_higher_sport_education    -0.1893      0.135     -1.398      0.163      -0.456       0.077
sport_master                   0.5302      0.164      3.239      0.001       0.208       0.852
sport_master_kandidate         0.2583      0.237      1.088      0.277      -0.209       0.725
is_champion                    0.4507      0.238      1.896      0.059      -0.017       0.919
exp_centered                  -0.0016      0.009     -0.174      0.862      -0.020       0.017
docs_verified                 -0.1006      0.146     -0.688      0.492      -0.388       0.187
rating                         0.0049      0.034      0.144      0.885      -0.062       0.072
reviews_count                 -0.0084      0.006     -1.317      0.189      -0.021       0.004
is_recomended                  0.1566      0.221      0.708      0.480      -0.279       0.592
photos_count                   0.0175      0.010      1.700      0.090      -0.003       0.038
is_male                       -0.1305      0.129     -1.014      0.311      -0.384       0.123
years_on_profi_ru              0.0230      0.025      0.929      0.354      -0.026       0.072
exp_centered2                 -0.0012      0.001     -1.748      0.082      -0.003       0.000
reputation_density             0.0473      0.040      1.195      0.233      -0.031       0.125
champion_experience_inter     -0.0302      0.034     -0.890      0.374      -0.097       0.037
is_outlier                     3.2527      0.347      9.364      0.000       2.569       3.936
==============================================================================
Omnibus:                       28.003   Durbin-Watson:                   2.077
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              101.858
Skew:                           0.226   Prob(JB):                     7.62e-23
Kurtosis:                       5.731   Cond. No.                         830.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Уберем переменную `rating`

In [43]:
cols_to_drop.append(worst_var)
cur_model_res = sm.OLS(y, X.drop(cols_to_drop, axis=1)).fit()

worst_p = cur_model_res.pvalues.drop('const').max()
worst_var = cur_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии текущей модели
print(f"Текущий AIC: {cur_model_res.aic:.2f} | BIC: {cur_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 977.42 | BIC: 1037.67
Худшая переменная: 'exp_centered' (p-value = 0.8502)
------------------------------


In [44]:
cur_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.331
Model:                            OLS   Adj. R-squared:                  0.297
Method:                 Least Squares   F-statistic:                     9.978
Date:                Thu, 07 May 2026   Prob (F-statistic):           2.96e-19
Time:                        19:09:00   Log-Likelihood:                -472.71
No. Observations:                 319   AIC:                             977.4
Df Residuals:                     303   BIC:                             1038.
Df Model:                          15                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.7873      0.214     13.052      0.000       2.367       3.208
has_higher_sport_education    -0.1880      0.135     -1.394      0.164      -0.453       0.077
sport_master                   0.5330      0.162      3.285      0.001       0.214       0.852
sport_master_kandidate         0.2594      0.237      1.095      0.274      -0.207       0.726
is_champion                    0.4500      0.237      1.896      0.059      -0.017       0.917
exp_centered                  -0.0018      0.009     -0.189      0.850      -0.020       0.017
docs_verified                 -0.0964      0.143     -0.674      0.501      -0.378       0.185
reviews_count                 -0.0086      0.006     -1.355      0.177      -0.021       0.004
is_recomended                  0.1578      0.221      0.715      0.475      -0.277       0.592
photos_count                   0.0176      0.010      1.705      0.089      -0.003       0.038
is_male                       -0.1314      0.128     -1.024      0.306      -0.384       0.121
years_on_profi_ru              0.0241      0.024      1.022      0.308      -0.022       0.070
exp_centered2                 -0.0012      0.001     -1.745      0.082      -0.003       0.000
reputation_density             0.0491      0.037      1.314      0.190      -0.024       0.123
champion_experience_inter     -0.0302      0.034     -0.889      0.374      -0.097       0.037
is_outlier                     3.2511      0.347      9.379      0.000       2.569       3.933
==============================================================================
Omnibus:                       28.032   Durbin-Watson:                   2.074
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              102.112
Skew:                           0.226   Prob(JB):                     6.71e-23
Kurtosis:                       5.735   Cond. No.                         829.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Итог: правильно убрали переменную `rating`, *AIC* и *BIC* упали

### Уберем переменную `exp_centered`

In [45]:
cols_to_drop.append(worst_var)
cur_model_res = sm.OLS(y, X.drop(cols_to_drop, axis=1)).fit()

worst_p = cur_model_res.pvalues.drop('const').max()
worst_var = cur_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии текущей модели
print(f"Текущий AIC: {cur_model_res.aic:.2f} | BIC: {cur_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 975.46 | BIC: 1031.94
Худшая переменная: 'docs_verified' (p-value = 0.5021)
------------------------------


In [46]:
cur_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.331
Model:                            OLS   Adj. R-squared:                  0.300
Method:                 Least Squares   F-statistic:                     10.72
Date:                Thu, 07 May 2026   Prob (F-statistic):           8.96e-20
Time:                        19:09:00   Log-Likelihood:                -472.73
No. Observations:                 319   AIC:                             975.5
Df Residuals:                     304   BIC:                             1032.
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.8013      0.200     14.002      0.000       2.408       3.195
has_higher_sport_education    -0.1934      0.132     -1.470      0.142      -0.452       0.065
sport_master                   0.5302      0.161      3.287      0.001       0.213       0.848
sport_master_kandidate         0.2606      0.236      1.102      0.271      -0.205       0.726
is_champion                    0.4525      0.237      1.913      0.057      -0.013       0.918
docs_verified                 -0.0960      0.143     -0.672      0.502      -0.377       0.185
reviews_count                 -0.0087      0.006     -1.397      0.163      -0.021       0.004
is_recomended                  0.1600      0.220      0.727      0.468      -0.273       0.593
photos_count                   0.0177      0.010      1.719      0.087      -0.003       0.038
is_male                       -0.1348      0.127     -1.063      0.289      -0.384       0.115
years_on_profi_ru              0.0235      0.023      1.007      0.315      -0.022       0.069
exp_centered2                 -0.0012      0.001     -1.785      0.075      -0.002       0.000
reputation_density             0.0500      0.037      1.350      0.178      -0.023       0.123
champion_experience_inter     -0.0302      0.034     -0.893      0.373      -0.097       0.036
is_outlier                     3.2513      0.346      9.394      0.000       2.570       3.932
==============================================================================
Omnibus:                       27.794   Durbin-Watson:                   2.075
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              101.005
Skew:                           0.221   Prob(JB):                     1.17e-22
Kurtosis:                       5.721   Cond. No.                         828.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Итог: правильно убрали переменную `exp_centered`, *AIC* и *BIC* упали

### Уберем переменную `docs_verified`

In [47]:
cols_to_drop.append(worst_var)
cur_model_res = sm.OLS(y, X.drop(cols_to_drop, axis=1)).fit()

worst_p = cur_model_res.pvalues.drop('const').max()
worst_var = cur_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии текущей модели
print(f"Текущий AIC: {cur_model_res.aic:.2f} | BIC: {cur_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 973.93 | BIC: 1026.65
Худшая переменная: 'is_recomended' (p-value = 0.4947)
------------------------------


In [48]:
cur_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.330
Model:                            OLS   Adj. R-squared:                  0.301
Method:                 Least Squares   F-statistic:                     11.53
Date:                Thu, 07 May 2026   Prob (F-statistic):           3.16e-20
Time:                        19:09:00   Log-Likelihood:                -472.97
No. Observations:                 319   AIC:                             973.9
Df Residuals:                     305   BIC:                             1027.
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.7513      0.186     14.829      0.000       2.386       3.116
has_higher_sport_education    -0.1933      0.131     -1.471      0.142      -0.452       0.065
sport_master                   0.5346      0.161      3.320      0.001       0.218       0.851
sport_master_kandidate         0.2780      0.235      1.184      0.237      -0.184       0.740
is_champion                    0.4460      0.236      1.888      0.060      -0.019       0.911
reviews_count                 -0.0085      0.006     -1.370      0.172      -0.021       0.004
is_recomended                  0.1500      0.219      0.684      0.495      -0.282       0.582
photos_count                   0.0169      0.010      1.654      0.099      -0.003       0.037
is_male                       -0.1353      0.127     -1.068      0.286      -0.385       0.114
years_on_profi_ru              0.0210      0.023      0.913      0.362      -0.024       0.066
exp_centered2                 -0.0012      0.001     -1.752      0.081      -0.002       0.000
reputation_density             0.0469      0.037      1.277      0.203      -0.025       0.119
champion_experience_inter     -0.0304      0.034     -0.898      0.370      -0.097       0.036
is_outlier                     3.2579      0.346      9.426      0.000       2.578       3.938
==============================================================================
Omnibus:                       27.302   Durbin-Watson:                   2.089
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               96.468
Skew:                           0.226   Prob(JB):                     1.13e-21
Kurtosis:                       5.656   Cond. No.                         828.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Итог: правильно убрали переменную `docs_verified`, *AIC* и *BIC* упали

### Уберем переменную `is_recomended`

In [49]:
cols_to_drop.append(worst_var)
cur_model_res = sm.OLS(y, X.drop(cols_to_drop, axis=1)).fit()

worst_p = cur_model_res.pvalues.drop('const').max()
worst_var = cur_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии текущей модели
print(f"Текущий AIC: {cur_model_res.aic:.2f} | BIC: {cur_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 972.42 | BIC: 1021.37
Худшая переменная: 'champion_experience_inter' (p-value = 0.3655)
------------------------------


In [50]:
cur_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.329
Model:                            OLS   Adj. R-squared:                  0.302
Method:                 Least Squares   F-statistic:                     12.48
Date:                Thu, 07 May 2026   Prob (F-statistic):           1.08e-20
Time:                        19:09:00   Log-Likelihood:                -473.21
No. Observations:                 319   AIC:                             972.4
Df Residuals:                     306   BIC:                             1021.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.7544      0.185     14.863      0.000       2.390       3.119
has_higher_sport_education    -0.1927      0.131     -1.467      0.143      -0.451       0.066
sport_master                   0.5358      0.161      3.331      0.001       0.219       0.852
sport_master_kandidate         0.2601      0.233      1.116      0.266      -0.199       0.719
is_champion                    0.4507      0.236      1.911      0.057      -0.013       0.915
reviews_count                 -0.0076      0.006     -1.247      0.213      -0.019       0.004
photos_count                   0.0168      0.010      1.648      0.100      -0.003       0.037
is_male                       -0.1366      0.127     -1.079      0.282      -0.386       0.113
years_on_profi_ru              0.0223      0.023      0.973      0.331      -0.023       0.067
exp_centered2                 -0.0012      0.001     -1.788      0.075      -0.002       0.000
reputation_density             0.0467      0.037      1.273      0.204      -0.026       0.119
champion_experience_inter     -0.0306      0.034     -0.906      0.366      -0.097       0.036
is_outlier                     3.2552      0.345      9.427      0.000       2.576       3.935
==============================================================================
Omnibus:                       26.870   Durbin-Watson:                   2.093
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               94.959
Skew:                           0.214   Prob(JB):                     2.40e-21
Kurtosis:                       5.638   Cond. No.                         828.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Итог: правильно убрали переменную `reputation_density`, *AIC* и *BIC* упали

### Уберем переменную `champion_experience_inter`

In [51]:
cols_to_drop.append(worst_var)
cur_model_res = sm.OLS(y, X.drop(cols_to_drop, axis=1)).fit()

worst_p = cur_model_res.pvalues.drop('const').max()
worst_var = cur_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии текущей модели
print(f"Текущий AIC: {cur_model_res.aic:.2f} | BIC: {cur_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 971.28 | BIC: 1016.46
Худшая переменная: 'years_on_profi_ru' (p-value = 0.5354)
------------------------------


In [52]:
cur_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.327
Model:                            OLS   Adj. R-squared:                  0.303
Method:                 Least Squares   F-statistic:                     13.54
Date:                Thu, 07 May 2026   Prob (F-statistic):           4.17e-21
Time:                        19:09:00   Log-Likelihood:                -473.64
No. Observations:                 319   AIC:                             971.3
Df Residuals:                     307   BIC:                             1016.
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.7882      0.181     15.366      0.000       2.431       3.145
has_higher_sport_education    -0.1765      0.130     -1.357      0.176      -0.432       0.079
sport_master                   0.5510      0.160      3.445      0.001       0.236       0.866
sport_master_kandidate         0.2726      0.233      1.172      0.242      -0.185       0.730
is_champion                    0.2903      0.156      1.863      0.063      -0.016       0.597
reviews_count                 -0.0079      0.006     -1.299      0.195      -0.020       0.004
photos_count                   0.0160      0.010      1.575      0.116      -0.004       0.036
is_male                       -0.1390      0.127     -1.098      0.273      -0.388       0.110
years_on_profi_ru              0.0125      0.020      0.620      0.535      -0.027       0.052
exp_centered2                 -0.0012      0.001     -1.770      0.078      -0.002       0.000
reputation_density             0.0485      0.037      1.323      0.187      -0.024       0.121
is_outlier                     3.2668      0.345      9.470      0.000       2.588       3.946
==============================================================================
Omnibus:                       27.372   Durbin-Watson:                   2.081
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              100.340
Skew:                           0.204   Prob(JB):                     1.63e-22
Kurtosis:                       5.717   Cond. No.                         828.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Итог: правильно убрали переменную `champion_experience_inter`, *AIC* и *BIC* упали

### Уберем переменную `years_on_profi_ru`

In [53]:
cols_to_drop.append(worst_var)
cur_model_res = sm.OLS(y, X.drop(cols_to_drop, axis=1)).fit()

worst_p = cur_model_res.pvalues.drop('const').max()
worst_var = cur_model_res.pvalues.drop('const').idxmax()
worst_p

##  критерии текущей модели
print(f"Текущий AIC: {cur_model_res.aic:.2f} | BIC: {cur_model_res.bic:.2f}")
print(f"Худшая переменная: '{worst_var}' (p-value = {worst_p:.4f})")
print("-" * 30)

Текущий AIC: 969.68 | BIC: 1011.09
Худшая переменная: 'is_male' (p-value = 0.3101)
------------------------------


In [54]:
cur_model_res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     price_in_thousands   R-squared:                       0.326
Model:                            OLS   Adj. R-squared:                  0.304
Method:                 Least Squares   F-statistic:                     14.89
Date:                Thu, 07 May 2026   Prob (F-statistic):           1.25e-21
Time:                        19:09:01   Log-Likelihood:                -473.84
No. Observations:                 319   AIC:                             969.7
Df Residuals:                     308   BIC:                             1011.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
==============================================================================================
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const                          2.8337      0.166     17.088      0.000       2.507       3.160
has_higher_sport_education    -0.1600      0.127     -1.258      0.209      -0.410       0.090
sport_master                   0.5701      0.157      3.636      0.000       0.262       0.879
sport_master_kandidate         0.2780      0.232      1.197      0.232      -0.179       0.735
is_champion                    0.2822      0.155      1.819      0.070      -0.023       0.588
reviews_count                 -0.0060      0.005     -1.142      0.254      -0.016       0.004
photos_count                   0.0164      0.010      1.621      0.106      -0.004       0.036
is_male                       -0.1270      0.125     -1.017      0.310      -0.373       0.119
exp_centered2                 -0.0013      0.001     -2.074      0.039      -0.003   -6.66e-05
reputation_density             0.0391      0.033      1.173      0.242      -0.026       0.105
is_outlier                     3.2634      0.345      9.471      0.000       2.585       3.941
==============================================================================
Omnibus:                       26.959   Durbin-Watson:                   2.085
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              101.172
Skew:                           0.175   Prob(JB):                     1.07e-22
Kurtosis:                       5.737   Cond. No.                         828.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Итог: правильно убрали переменную `years_on_profi_ru`, *AIC* и *BIC* упали

## пока стоп, остановимся с удалением фичей, сделаем тесты на спецификацию и еще раз сделаем центривание `work_experience`

In [55]:
data_left = X.drop(cols_to_drop, axis=1)

In [56]:
rest_features = data_left.columns
rest_features

Index(['const', 'has_higher_sport_education', 'sport_master',
       'sport_master_kandidate', 'is_champion', 'reviews_count',
       'photos_count', 'is_male', 'exp_centered2', 'reputation_density',
       'is_outlier'],
      dtype='object')